# COMPLETE MACHINE LEARNING JOURNEY: Baseline → Advanced CNN
  Dataset: MNIST Handwritten Digits (loaded via TensorFlow/Keras — no manual download)
  Libraries: NumPy, Pandas, Matplotlib, Seaborn, Scikit-Learn, TensorFlow/Keras

TABLE OF CONTENTS
-----------------
 1. Setup & Imports
 2. Data Loading & Exploration (EDA)
 3. Data Preprocessing
 4. BASELINE MODEL — Logistic Regression
 5. IMPROVEMENT 1 — Decision Tree
 6. IMPROVEMENT 2 — Random Forest
 7. IMPROVEMENT 3 — Support Vector Machine (SVM)
 8. IMPROVEMENT 4 — Basic Neural Network (MLP via Keras)
 9. IMPROVEMENT 5 — Deeper MLP with Batch Normalization & Dropout
10. IMPROVEMENT 6 — Convolutional Neural Network (CNN)
11. IMPROVEMENT 7 — Advanced CNN with Data Augmentation
12. Final Comparison Dashboard

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend (safe for scripts)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import warnings
import time
import os

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
# Scikit-Learn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from sklearn.decomposition import PCA

In [ ]:
# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
print("=" * 70)
print("  ML JOURNEY: MNIST Handwritten Digit Classification")
print("=" * 70)
print(f"\n  TensorFlow version : {tf.__version__}")
print(f"  NumPy version      : {np.__version__}")
print(f"  Pandas version     : {pd.__version__}")
 

In [ ]:
OUTPUT_DIR = "ml_journey_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Palette used throughout
PALETTE     = "viridis"
ACCENT      = "#4C72B0"
SUCCESS_CLR = "#2ecc71"
DANGER_CLR  = "#e74c3c"

# SECTION 2: DATA LOADING & EXPLORATORY DATA ANALYSIS

In [ ]:
print("\n" + "=" * 70)
print("  SECTION 2 — Data Loading & Exploratory Data Analysis")
print("=" * 70)

In [ ]:
#  Load MNIST 
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.mnist.load_data()
 
print(f"\n  Raw training set  : {X_train_raw.shape}  labels: {y_train_raw.shape}")
print(f"  Raw test set      : {X_test_raw.shape}  labels: {y_test_raw.shape}")
 

In [ ]:
# Convert to DataFrame for Pandas EDA
n_pixels = 28 * 28
X_train_flat_raw = X_train_raw.reshape(-1, n_pixels)
X_test_flat_raw  = X_test_raw.reshape(-1, n_pixels)
 
col_names  = [f"pixel_{i}" for i in range(n_pixels)]
df_train   = pd.DataFrame(X_train_flat_raw, columns=col_names)
df_train["label"] = y_train_raw

print("\n  ── Pandas DataFrame (first 5 rows, first 10 pixel cols + label) ──")
print(df_train[col_names[:10] + ["label"]].head())
 
print("\n  ── Basic Statistics (pixel_0 … pixel_9) ──")
print(df_train[col_names[:10]].describe().round(2))
 
print("\n  ── Class Distribution ──")
class_dist = df_train["label"].value_counts().sort_index()
print(class_dist.to_string())

In [ ]:
# EDA Figure 1: Sample images + class distribution

fig = plt.figure(figsize=(20, 10))
fig.patch.set_facecolor("#1a1a2e")
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

In [ ]:
#  Sample images
ax_imgs = fig.add_subplot(gs[0, 0])
ax_imgs.set_facecolor("#1a1a2e")
ax_imgs.axis("off")
ax_imgs.set_title("Sample Images (one per class)", color="white", fontsize=14, pad=10)
for digit in range(10):
    idx = np.where(y_train_raw == digit)[0][0]
    ax_inner = fig.add_axes([0.03 + digit * 0.047, 0.55, 0.042, 0.1])
    ax_inner.imshow(X_train_raw[idx], cmap="plasma")
    ax_inner.set_title(str(digit), color="white", fontsize=9)
    ax_inner.axis("off")
 

In [ ]:
#  Class distribution bar
ax_bar = fig.add_subplot(gs[0, 1])
ax_bar.set_facecolor("#16213e")
colors = sns.color_palette("plasma", 10)
bars   = ax_bar.bar(class_dist.index, class_dist.values, color=colors, edgecolor="white", linewidth=0.5)
ax_bar.set_title("Class Distribution (Training Set)", color="white", fontsize=13)
ax_bar.set_xlabel("Digit", color="#aaaaaa")
ax_bar.set_ylabel("Count", color="#aaaaaa")
ax_bar.tick_params(colors="white")
for spine in ax_bar.spines.values():
    spine.set_edgecolor("#444466")
for bar, cnt in zip(bars, class_dist.values):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                f"{cnt:,}", ha="center", va="bottom", color="white", fontsize=8)
 

In [ ]:
# Pixel intensity histogram
ax_hist = fig.add_subplot(gs[1, 0])
ax_hist.set_facecolor("#16213e")
sample_pixels = X_train_flat_raw[:5000].flatten()
ax_hist.hist(sample_pixels, bins=50, color=ACCENT, edgecolor="none", alpha=0.85)
ax_hist.set_title("Pixel Intensity Distribution (5k samples)", color="white", fontsize=13)
ax_hist.set_xlabel("Pixel Value (0–255)", color="#aaaaaa")
ax_hist.set_ylabel("Frequency", color="#aaaaaa")
ax_hist.tick_params(colors="white")
for spine in ax_hist.spines.values():
    spine.set_edgecolor("#444466")

In [ ]:
#  Mean digit images
ax_mean = fig.add_subplot(gs[1, 1])
ax_mean.axis("off")
ax_mean.set_facecolor("#1a1a2e")
ax_mean.set_title("Mean Image per Class", color="white", fontsize=13, pad=10)
for digit in range(10):
    mask       = y_train_raw == digit
    mean_img   = X_train_raw[mask].mean(axis=0)
    ax_m = fig.add_axes([0.54 + (digit % 5) * 0.087,
                         0.08 + (1 - digit // 5) * 0.19,
                         0.075, 0.15])
    ax_m.imshow(mean_img, cmap="inferno")
    ax_m.set_title(f"μ({digit})", color="white", fontsize=8)
    ax_m.axis("off")
 
fig.suptitle("MNIST — Exploratory Data Analysis", color="white",
             fontsize=18, fontweight="bold", y=0.98)
path_eda = os.path.join(OUTPUT_DIR, "01_eda.png")
plt.savefig(path_eda, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"\n  [✓] EDA plot saved → {path_eda}")
 

In [ ]:
# EDA Figure 2: Correlation heatmap on PCA-reduced features
print("\n  Computing PCA (20 components) for correlation heatmap …")
pca_eda   = PCA(n_components=20, random_state=SEED)
X_pca_eda = pca_eda.fit_transform(X_train_flat_raw[:10000] / 255.0)
df_pca    = pd.DataFrame(X_pca_eda, columns=[f"PC{i+1}" for i in range(20)])
df_pca["label"] = y_train_raw[:10000]
 
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor("#1a1a2e")

ax1 = axes[0]
ax1.set_facecolor("#16213e")
corr = df_pca.drop("label", axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0,
            annot=False, linewidths=0.3, ax=ax1,
            cbar_kws={"shrink": 0.8})
ax1.set_title("PCA Component Correlation Matrix", color="white", fontsize=13, pad=10)
ax1.tick_params(colors="white", labelsize=8)


ax2 = axes[1]
ax2.set_facecolor("#16213e")
explained = pca_eda.explained_variance_ratio_ * 100
cumulative = np.cumsum(explained)
ax2.bar(range(1, 21), explained, color=sns.color_palette("plasma", 20), label="Individual")
ax2.plot(range(1, 21), cumulative, "w-o", linewidth=2, markersize=5, label="Cumulative")
ax2.axhline(80, color=SUCCESS_CLR, linestyle="--", linewidth=1.5, label="80% threshold")
ax2.set_title("PCA Explained Variance", color="white", fontsize=13)
ax2.set_xlabel("Principal Component", color="#aaaaaa")
ax2.set_ylabel("Variance Explained (%)", color="#aaaaaa")
ax2.tick_params(colors="white")
legend = ax2.legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")
 
for spine in ax2.spines.values():
    spine.set_edgecolor("#444466")
 
fig.suptitle("MNIST — PCA Analysis", color="white", fontsize=16, fontweight="bold")
path_pca = os.path.join(OUTPUT_DIR, "02_pca_analysis.png")
plt.savefig(path_pca, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"  [✓] PCA analysis plot saved → {path_pca}")


# SECTION 3: DATA PREPROCESSING


In [ ]:
print("\n" + "=" * 70)
print("  SECTION 3 — Data Preprocessing")
print("=" * 70)


In [ ]:
# Normalize pixel values to [0, 1] 
X_train_norm = X_train_flat_raw / 255.0    # shape: (60000, 784)
X_test_norm  = X_test_flat_raw  / 255.0    # shape: (10000, 784)

In [ ]:
# 2-D versions for CNN 
X_train_2d = X_train_raw.reshape(-1, 28, 28, 1) / 255.0
X_test_2d  = X_test_raw.reshape(-1, 28, 28, 1)  / 255.0
 


In [ ]:
# One-hot labels for Keras 
y_train_oh = keras.utils.to_categorical(y_train_raw, 10)
y_test_oh  = keras.utils.to_categorical(y_test_raw,  10)

In [ ]:
# Subsample for slower sklearn models
SKLEARN_TRAIN = 20_000
SKLEARN_TEST  = 5_000
 
X_sk_train = X_train_norm[:SKLEARN_TRAIN]
y_sk_train = y_train_raw[:SKLEARN_TRAIN]
X_sk_test  = X_test_norm[:SKLEARN_TEST]
y_sk_test  = y_test_raw[:SKLEARN_TEST]

print(f"\n  Normalized pixel range : [{X_train_norm.min():.2f}, {X_train_norm.max():.2f}]")
print(f"  Sklearn training size  : {SKLEARN_TRAIN:,}  test size: {SKLEARN_TEST:,}")
print(f"  Keras training size    : {X_train_norm.shape[0]:,}  test size: {X_test_norm.shape[0]:,}")

In [ ]:
# Storage for final comparison
results = {}

In [ ]:
#  HELPER FUNCTIONS
def evaluate_sklearn(name, model, X_tr, y_tr, X_te, y_te):
    """Train, time, evaluate, and store a scikit-learn model."""
    print(f"\n  Training {name} …")
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0
 
    y_pred = model.predict(X_te)
    acc    = accuracy_score(y_te, y_pred)
    report = classification_report(y_te, y_pred, output_dict=True)
    cm     = confusion_matrix(y_te, y_pred)
 
    results[name] = {"accuracy": acc, "train_time": train_time,
                     "history": None, "type": "sklearn"}
    print(f"  {name:40s} | Accuracy: {acc*100:.2f}% | Time: {train_time:.1f}s")
    return y_pred, cm, report

In [ ]:
def plot_confusion_matrix(cm, title, filepath, cmap="Blues"):
    """Styled confusion matrix heatmap."""
    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#16213e")
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap=cmap,
                linewidths=0.4, linecolor="#2a2a4a",
                xticklabels=range(10), yticklabels=range(10),
                cbar_kws={"label": "% of True Class"}, ax=ax)
    ax.set_title(title, color="white", fontsize=14, pad=12)
    ax.set_xlabel("Predicted Label", color="#aaaaaa")
    ax.set_ylabel("True Label", color="#aaaaaa")
    ax.tick_params(colors="white")
    plt.colorbar(ax.collections[0], ax=ax).ax.yaxis.label.set_color("white")
    plt.tight_layout()
    plt.savefig(filepath, dpi=130, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()

In [ ]:
def plot_keras_history(history, title, filepath):
    """Two-panel loss + accuracy plot for Keras training."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor("#1a1a2e")
 
    for ax, metric, ylabel in zip(
        axes, ["loss", "accuracy"], ["Loss", "Accuracy"]
    ):
        ax.set_facecolor("#16213e")
        ax.plot(history.history[metric],
                color=ACCENT, linewidth=2.5, label="Train")
        ax.plot(history.history[f"val_{metric}"],
                color=SUCCESS_CLR, linewidth=2.5,
                linestyle="--", label="Validation")
        ax.set_title(f"{title} — {ylabel}", color="white", fontsize=13)
        ax.set_xlabel("Epoch", color="#aaaaaa")
        ax.set_ylabel(ylabel, color="#aaaaaa")
        ax.tick_params(colors="white")
        legend = ax.legend(facecolor="#2a2a4a",
                           edgecolor="white", labelcolor="white")
        for spine in ax.spines.values():
            spine.set_edgecolor("#444466")
 
    plt.tight_layout()
    plt.savefig(filepath, dpi=130, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
 

In [ ]:
def show_misclassified(X_flat, y_true, y_pred, title, filepath, n=20):
    """Grid of misclassified examples."""
    wrong_idx = np.where(y_true != y_pred)[0][:n]
    cols, rows = 10, 2
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5))
    fig.patch.set_facecolor("#1a1a2e")
    fig.suptitle(title, color="white", fontsize=14, y=1.01)
    for i, idx in enumerate(wrong_idx):
        ax = axes[i // cols][i % cols]
        ax.imshow(X_flat[idx].reshape(28, 28), cmap="plasma")
        ax.set_title(f"T:{y_true[idx]} P:{y_pred[idx]}",
                     color=DANGER_CLR, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(filepath, dpi=130, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()

# SECTION 4: BASELINE — LOGISTIC REGRESSION

In [ ]:
print("\n" + "=" * 70)
print("  SECTION 4 — Baseline: Logistic Regression")
print("=" * 70)
 
lr_model = LogisticRegression(
    max_iter=1000, solver="lbfgs", multi_class="multinomial",
    C=1.0, random_state=SEED
)
lr_pred, lr_cm, lr_report = evaluate_sklearn(
    "Logistic Regression", lr_model,
    X_sk_train, y_sk_train, X_sk_test, y_sk_test
)
plot_confusion_matrix(
    lr_cm, "Logistic Regression — Confusion Matrix",
    os.path.join(OUTPUT_DIR, "03_lr_confusion.png"), cmap="Blues"
)
show_misclassified(
    X_sk_test, y_sk_test, lr_pred,
    "Logistic Regression — Misclassified Examples",
    os.path.join(OUTPUT_DIR, "04_lr_misclassified.png")
)

# Per-class bar chart
df_lr = pd.DataFrame(lr_report).T
df_lr = df_lr[df_lr.index.isin([str(i) for i in range(10)])]
 
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#16213e")
x    = np.arange(10)
w    = 0.25
bars = [
    ax.bar(x - w,     df_lr["precision"].astype(float), w, label="Precision", color="#4C72B0"),
    ax.bar(x,         df_lr["recall"].astype(float),    w, label="Recall",    color="#DD8452"),
    ax.bar(x + w,     df_lr["f1-score"].astype(float),  w, label="F1-Score",  color="#55A868"),
]
ax.set_xticks(x)
ax.set_xticklabels(range(10), color="white")
ax.set_title("Logistic Regression — Per-Class Metrics", color="white", fontsize=13)
ax.set_xlabel("Digit", color="#aaaaaa")
ax.set_ylabel("Score", color="#aaaaaa")
ax.tick_params(colors="white")
ax.legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")
ax.set_ylim(0, 1.1)
for spine in ax.spines.values():
    spine.set_edgecolor("#444466")
path_lr_metrics = os.path.join(OUTPUT_DIR, "05_lr_per_class.png")
plt.tight_layout()
plt.savefig(path_lr_metrics, dpi=130, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  [✓] LR per-class metrics saved → {path_lr_metrics}")
 





# SECTION 5: IMPROVEMENT 1 — DECISION TREE


In [ ]:
print("\n" + "=" * 70)
print("  SECTION 5 — Improvement 1: Decision Tree")
print("=" * 70)
 
dt_model = DecisionTreeClassifier(
    max_depth=20, min_samples_split=5,
    criterion="gini", random_state=SEED
)